# PCU-KILL-001 — Granite Engineering E0 on Kaggle

This notebook runs the **engineering-only** PCU composability kill test on
`ibm-granite/granite-3.1-1b-a400m-base`, records the exact engineering evidence,
and publishes only lightweight evidence back to `codex/pcu-composability-kill-001`.

Safety boundary:

- Engineering seed: `26090501`.
- Formal seeds `26090511 / 26090512 / 26090513` must remain `RESERVED_UNTOUCHED`.
- This notebook contains no formal execution or freeze command.
- Negative engineering outcomes are preserved and published; they are evidence.
- Binary fork/cache/model artifacts remain in Kaggle and are not committed to Git.
- Kaggle Secrets required: `HF_TOKEN` and `GITHUB_TOKEN`.
- Kaggle Internet must be **ON** and a GPU accelerator must be enabled.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import os
import shutil
import subprocess
import sys

BRANCH = "codex/pcu-composability-kill-001"
REPO = Path("/kaggle/working/mini-cells")
OUT = REPO / "artifacts/research/pcu-kill-001/engineering/26090501"
MODEL_ID = "ibm-granite/granite-3.1-1b-a400m-base"
ENGINEERING_SEED = 26090501
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = "5.16.1"

os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run(cmd, *, env=None, capture=False):
    cmd = [str(item) for item in cmd]
    print("+", " ".join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ""

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])
os.chdir(REPO)
run(["git", "pull", "--ff-only", "origin", BRANCH])

assert sys.version_info >= (3, 11), f"MiniCells requires Python >=3.11, got {sys.version.split()[0]}"

# Keep Kaggle's PyTorch; install repo/test tooling and the modern fused Granite-MoE runtime.
run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])
run([
    sys.executable, "-m", "pip", "install",
    f"transformers=={REQUIRED_TRANSFORMERS}",
    "huggingface_hub>=0.36,<2.0",
    "safetensors>=0.4",
    "accelerate>=1.0",
])

import torch
import transformers

assert transformers.__version__ == REQUIRED_TRANSFORMERS, (
    f"Expected transformers {REQUIRED_TRANSFORMERS}, got {transformers.__version__}. "
    "Restart the Kaggle kernel and Run All if an older module was already loaded."
)
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."

SOURCE_COMMIT = run(["git", "rev-parse", "HEAD"], capture=True)
SOURCE_TREE = run(["git", "rev-parse", "HEAD^{tree}"], capture=True)
print(json.dumps({
    "branch": run(["git", "branch", "--show-current"], capture=True),
    "commit": SOURCE_COMMIT,
    "tree": SOURCE_TREE,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": version("huggingface_hub"),
    "cuda": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
github_token = secrets.get_secret("GITHUB_TOKEN")
assert hf_token, "Missing Kaggle Secret: HF_TOKEN"
assert github_token, "Missing Kaggle Secret: GITHUB_TOKEN"

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["GITHUB_TOKEN"] = github_token
login(token=hf_token, add_to_git_credential=False)
print("Kaggle Secrets loaded; token values were not printed.")


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoConfig

api = HfApi(token=os.environ["HF_TOKEN"])
model_info = api.model_info(MODEL_ID)
HF_PREFLIGHT_REVISION = model_info.sha
assert HF_PREFLIGHT_REVISION and len(HF_PREFLIGHT_REVISION) >= 7

cfg = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=HF_PREFLIGHT_REVISION,
    token=os.environ["HF_TOKEN"],
)
assert cfg.model_type == "granitemoe", cfg.model_type
assert "GraniteMoeForCausalLM" in list(getattr(cfg, "architectures", []) or [])
assert int(cfg.hidden_size) == 1024
assert int(cfg.intermediate_size) == 512
assert int(cfg.num_hidden_layers) == 24
assert int(cfg.num_local_experts) == 32
assert int(cfg.num_experts_per_tok) == 8

print(json.dumps({
    "model": MODEL_ID,
    "hf_preflight_revision": HF_PREFLIGHT_REVISION,
    "checkpoint_transformers_version": getattr(cfg, "transformers_version", None),
    "runtime_transformers_version": REQUIRED_TRANSFORMERS,
    "hidden_size": cfg.hidden_size,
    "intermediate_size": cfg.intermediate_size,
    "layers": cfg.num_hidden_layers,
    "experts": cfg.num_local_experts,
    "top_k": cfg.num_experts_per_tok,
}, indent=2))
print(
    "Checkpoint metadata records its original Transformers version; "
    "PCU uses the modern fused Granite-MoE runtime and proves equivalence through G0."
)


In [ ]:
SEED_REGISTRY = REPO / "research/formal_seed_registry.json"

def assert_formal_registry_untouched():
    payload = json.loads(SEED_REGISTRY.read_text(encoding="utf-8"))
    states = {int(row["seed"]): row["state"] for row in payload["seeds"]}
    expected = {seed: "RESERVED_UNTOUCHED" for seed in FORMAL_SEEDS}
    assert states == expected, (states, expected)
    return states

states = assert_formal_registry_untouched()
assert run(["git", "diff", "--", str(SEED_REGISTRY.relative_to(REPO))], capture=True) == ""

tracked = subprocess.run(
    ["git", "ls-files", "--error-unmatch",
     str((OUT / "ENGINEERING_DECISION.json").relative_to(REPO))],
    cwd=REPO, text=True, capture_output=True,
)
assert tracked.returncode != 0, (
    "Canonical PCU-KILL-001 engineering E0 evidence is already tracked on this branch. "
    "Refusing to rerun/overwrite it."
)
assert not (REPO / "artifacts/research/pcu-kill-001/formal").exists()

print(json.dumps({
    "engineering_seed": ENGINEERING_SEED,
    "formal_seed_states": states,
    "formal_execution_not_started": True,
    "canonical_e0_already_tracked": False,
}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(REPO / "src")
test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/research/05-pcu-kill-001",
], env=test_env)
run([
    sys.executable, "-m", "compileall", "-q",
    "src/minicells/pcu_kill_001",
    "scripts/research",
])
print("PCU-KILL-001 test/compile gate: PASS")


In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)

run([
    sys.executable,
    "scripts/research/run_pcu_kill_001.py",
    "--phase", "engineering",
    "--backend", "granite",
    "--seed", str(ENGINEERING_SEED),
    "--device", "cuda",
    "--out", OUT,
])


In [ ]:
decision_path = OUT / "ENGINEERING_DECISION.json"
assert decision_path.is_file(), (
    "Granite E0 did not produce ENGINEERING_DECISION.json. "
    "Treat this as an execution/environment failure, not a scientific result."
)
decision = json.loads(decision_path.read_text(encoding="utf-8"))

assert decision.get("phase") == "engineering"
assert decision.get("scientific_evidence") is False
assert decision.get("formal_execution_not_started") is True
assert decision.get("status") not in {
    None,
    "REAL_GRANITE_E0_NOT_RUN",
    "FORMAL_EXECUTION_FAILED",
}

summary = {
    "status": decision.get("status"),
    "valid_run": decision.get("valid_run"),
    "formal_ready": decision.get("formal_ready"),
    "gates": decision.get("gates"),
    "metrics": decision.get("metrics"),
    "output": str(OUT),
}
print("=== PCU-KILL-001 ENGINEERING E0 ===")
print(json.dumps(summary, indent=2, default=str))

for name in [
    "MODEL_MANIFEST.json",
    "EQUIVALENCE.json",
    "CACHE_EQUIVALENCE.json",
    "DATASET_AUDIT.json",
    "CONTEXT_ORACLE.json",
    "CAPACITY_LADDER.json",
    "METRICS_PCU.json",
    "METRICS_LORA.json",
    "MERGE_AUDIT.json",
    "ENGINEERING_DECISION.json",
    "QA_LOG.md",
]:
    path = OUT / name
    if path.exists():
        print(f"evidence: {name} ({path.stat().st_size} bytes)")

model_manifest_path = OUT / "MODEL_MANIFEST.json"
if model_manifest_path.exists():
    model_manifest = json.loads(model_manifest_path.read_text(encoding="utf-8"))
    print("Runner-pinned model revision:", model_manifest.get("model_revision"))
    print("HF preflight model revision:", HF_PREFLIGHT_REVISION)


In [ ]:
states_after = assert_formal_registry_untouched()
assert run(["git", "diff", "--", str(SEED_REGISTRY.relative_to(REPO))], capture=True) == ""
assert not (REPO / "artifacts/research/pcu-kill-001/formal").exists()
assert not (REPO / "artifacts/research/pcu-kill-001/frozen").exists(), (
    "Engineering notebook unexpectedly created frozen protocol artifacts."
)

print(json.dumps({
    "formal_seed_states_after_e0": states_after,
    "formal_execution_not_started": True,
    "engineering_status": decision["status"],
}, indent=2))


In [ ]:
run([
    sys.executable,
    "scripts/research/publish_pcu_kill_001_engineering.py",
    "--branch", BRANCH,
    "--token-env", "GITHUB_TOKEN",
])

published_head = run(["git", "rev-parse", "HEAD"], capture=True)
print(json.dumps({
    "published": True,
    "branch": BRANCH,
    "commit": published_head,
    "engineering_seed": ENGINEERING_SEED,
    "engineering_status": decision["status"],
    "formal_execution_not_started": True,
}, indent=2))
